# Lab Task – Simple RAG System for Tips Hindawi University

This notebook implements a bilingual RAG pipeline for answering questions
using information retrieved from a university information PDF.

Pipeline:

1. Install/import libraries.
2. Load an instruction-tuned causal language model.
3. Define a text-generation function.
4. Extract text from the PDF.
5. Split the PDF text into chunks.
6. Convert chunks to multilingual embeddings.
7. Build a FAISS vector index.
8. Retrieve the most relevant chunks for a question.
9. Put the retrieved context into a prompt.
10. Generate the final answer.

**Bilingual design:** Qwen2.5-3B-Instruct is used as the generator and
`intfloat/multilingual-e5-base` as the embedding model.

The pipeline supports both English and Arabic questions and documents.
For this experiment, an Arabic version of the university information PDF
is used as the knowledge source.

In [1]:
# Install the same main libraries used by the lecture notebook.
# Qwen2.5-3B-Instruct is used instead of Mistral-Nemo because we want
# one generator that can work well with both English and Arabic.

!pip install -q transformers==4.52.4 sentence-transformers PyPDF2 faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 57.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.1 MB/s eta 0:00:00:00:01


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import torch
import faiss
import os

In [3]:
# Bilingual instruction-following generator.
# Qwen2.5-3B-Instruct

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
def generate_text(prompt, max_new_tokens=350):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a retrieval-augmented question answering assistant. "
                "Answer ONLY using the information provided in the retrieved context. "
                "Important rules:\n"
                "- Answer in the same language as the question.\n"
                "- Give a complete answer.\n"
                "- Include ALL relevant facts from the context.\n"
                "- Do NOT stop after mentioning only the first fact.\n"
                "- Preserve important numbers, measurements, locations, names, and dates.\n"
                "- Do not invent or add information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [5]:
# PDF text extraction – same idea as the lecture notebook.

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""

    for page in reader.pages:
        page_text = page.extract_text() or ""
        full_text += page_text + "\n"

    return full_text

In [6]:
# Text chunking – same word-based approach as the lecture notebook.
# The smaller chunk size is useful for this relatively short university PDF.

def chunk_text(text, chunk_size=150, overlap=40):
    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

    return chunks

In [7]:
# Multilingual embeddings.
# multilingual-e5-base supports many languages, including Arabic and English.

def embed_chunks(chunks, model_name="intfloat/multilingual-e5-base"):
    embedding_model = SentenceTransformer(model_name)

    # E5 recommends "passage:" for stored document chunks.
    passage_texts = [f"passage: {chunk}" for chunk in chunks]

    embeddings = embedding_model.encode(
        passage_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embedding_model, embeddings

In [8]:
def create_faiss_index(embeddings):
    # Because embeddings are normalized, inner product is equivalent
    # to cosine similarity.

    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    return index

In [9]:
def search_index(query, model, index, chunks, k=5):

    query_embedding = model.encode(
        [f"query: {query}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        min(k, len(chunks))
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), 1
    ):
        if idx != -1:
            results.append({
                "rank": rank,
                "chunk_id": int(idx),
                "chunk": chunks[idx],
                "score": float(score)
            })

    return results

In [10]:
# Path to the uploaded PDF.

pdf_path = "/kaggle/input/datasets/andrewsameh20/tips-hindawi-university-arabic-information/.pdf"

text = extract_text_from_pdf(pdf_path)

chunks = chunk_text(
    text,
    chunk_size=100,
    overlap=20
)

model_embeddings, embeddings = embed_chunks(chunks)

index = create_faiss_index(embeddings)

print("Number of characters:", len(text))
print("Number of chunks:", len(chunks))
print("Embedding dimension:", embeddings.shape[1])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Number of characters: 4928
Number of chunks: 10
Embedding dimension: 768


In [11]:
# Optional inspection: print the first few chunks.

for i, chunk in enumerate(chunks[:5], 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
1. نظرة عامة جامعة هنداوي (THU) هي مؤسسة رائدة للتعليم العالي تقع في قلب منطقة الشرق األوسط. تأسست الجامعة عام 1963 ، ونمت لتصبحمركزًا معترفًا به عالميًا للتميز األكاديمي واالبتكار . وعلى مدار أكثر من ستة عقود من الريادة التعليمية، خرّجت الجامعة أكثر من 150,000 خريج يعملون في مختلف الصناعات والمجاالت األكاديمية حول العالم. الجامعة معتمدة من اللجنة الدولية للمعايير األكاديمية ووزارة التعليم العالي . وتعمل الجامعة وفق شعارها الموجّه: "المعرفة، النزاهة، التقدم " كما توفر بيئة تشجع على البحث العلمي واإلبداع وخدمة المجتمع . 2. الحرم الجامعي والمرافق 2.1 الحرم الجامعي الرئيسي يقع الحرم الجامعي الرئيسي في العاصمة، ويمتد

--- Chunk 2 ---
العلمي واإلبداع وخدمة المجتمع . 2. الحرم الجامعي والمرافق 2.1 الحرم الجامعي الرئيسي يقع الحرم الجامعي الرئيسي في العاصمة، ويمتد على مساحة تزيد عن 300 فدان. ويضم: • 12 مبنى أكاديميًا • مكتبة مركزية تحتوي على أكثر من مليون مجلد • أربعة مبانٍ سكنية للطالب • مركزًا طبيًا • مجمعًا رياضيًا واستادًا • مركزًا لالبتكار وريادة األعمال 2.2 الفروع الجامعي

In [12]:
def answer_question(question, k=5, max_new_tokens=350):

    top_chunks = search_index(
        question,
        model_embeddings,
        index,
        chunks,
        k=k
    )

    context = "\n\n".join(
        [
            f"[Retrieved Chunk {item['rank']}]\n{item['chunk']}"
            for item in top_chunks
        ]
    )

    prompt = f"""
Answer the following question using ONLY the retrieved context.

Question:
{question}

Retrieved Context:
{context}

Rules:
1. Use ONLY information explicitly mentioned in the retrieved context.
2. Give a complete answer.
3. Include ALL facts that directly answer the question.
4. If the answer contains multiple facts, include all of them.
5. Do NOT stop after the first fact.
6. Preserve important numbers, measurements, locations, names, and dates.
7. Do NOT remove important details when summarizing.
8. Do NOT use outside knowledge.
9. Do NOT invent information.
10. Answer in the same language as the question.
11. If the information is not supported by the context, say:
"المعلومة غير مذكورة في المستند."

Answer:
"""

    answer = generate_text(
        prompt,
        max_new_tokens=max_new_tokens
    )

    return answer, top_chunks, prompt

## Questions Required by the Lab

In [13]:
questions = [
    "أين تقع جامعة هنداوي؟",
    "هل تقدم الجامعة برامج دراسية عبر الإنترنت؟",
    "هل توجد مساعدات مالية للطلاب الدوليين؟",
    "ما هي اللغات المستخدمة في التدريس؟"
]

for q in questions:

    answer, top_chunks, prompt = answer_question(
        q,
        k=5
    )

    print("\n" + "=" * 100)
    print("السؤال:")
    print(q)

    print("-" * 100)
    print("الإجابة:")
    print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



السؤال:
أين تقع جامعة هنداوي؟
----------------------------------------------------------------------------------------------------
الإجابة:
جامعة هنداوي تقع في العاصمة. الحرم الجامعي الرئيسي يمتد على مساحة تزيد عن 300 فدان. يضم: 12 مبنى أكاديميًا، مكتبة مركزية تحتوي على أكثر من مليون مجلد، أربعة مبانٍ سكنية للطالب، مركزًا طبيًا، مجمعًا رياضيًا واستادًا، ومركزًا للابتكار وريادة الأعمال.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



السؤال:
هل تقدم الجامعة برامج دراسية عبر الإنترنت؟
----------------------------------------------------------------------------------------------------
الإجابة:
نعم، تقدم الجامعة برامج دراسية عبر الإنترنت. حيث أن هناك إشارة واضحة إلى تنظيم شبكة خريجي الجامعة ندوات عبر الإنترنت كل ثالثة أشهر، وهذا يشير إلى تقديم الجامعة برامج دراسية عبر الإنترنت.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



السؤال:
هل توجد مساعدات مالية للطلاب الدوليين؟
----------------------------------------------------------------------------------------------------
الإجابة:
نعم، يوجد مساعدات مالية للطلاب الدوليين. يتم تقديم منح قائمة على الحاجة المالية للطلاب الدوليين الذين يحتاجون إلى مساعدة مالية. كما يتم تقديم منح التفوق التي تغطي من 25% إلى 100% من الرسوم الدراسية.

السؤال:
ما هي اللغات المستخدمة في التدريس؟
----------------------------------------------------------------------------------------------------
الإجابة:
المعلومات المذكورة في المستند لا تشير إلى اللغات المستخدمة في التدريس بشكل مباشر. لذا، المعلومة غير مذكورة في المستند.


In [14]:
# Inspect retrieved evidence for each question.
# This is useful for demonstrating that the generator is grounded in the PDF.
for q in questions:

    answer, top_chunks, prompt = answer_question(
        q,
        k=5
    )

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(q)
    print("=" * 100)

    print("\nFINAL ANSWER")
    print("-" * 100)
    print(answer)

    print("\nRETRIEVED EVIDENCE")
    print("-" * 100)

    for item in top_chunks:

        print(
            f"\nRank: {item['rank']} | "
            f"Chunk ID: {item['chunk_id']} | "
            f"Similarity Score: {item['score']:.4f}"
        )

        print("-" * 100)
        print(item["chunk"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QUESTION:
أين تقع جامعة هنداوي؟

FINAL ANSWER
----------------------------------------------------------------------------------------------------
جامعة هنداوي تقع في العاصمة. الحرم الجامعي الرئيسي يمتد على مساحة تزيد عن 300 فدان. يضم: 12 مبنى أكاديميًا، مكتبة مركزية تحتوي على أكثر من مليون مجلد، أربعة مبانٍ سكنية للطالب، مركزًا طبيًا، مجمعًا رياضيًا واستادًا، ومركزًا للابتكار وريادة الأعمال.

RETRIEVED EVIDENCE
----------------------------------------------------------------------------------------------------

Rank: 1 | Chunk ID: 0 | Similarity Score: 0.8760
----------------------------------------------------------------------------------------------------
1. نظرة عامة جامعة هنداوي (THU) هي مؤسسة رائدة للتعليم العالي تقع في قلب منطقة الشرق األوسط. تأسست الجامعة عام 1963 ، ونمت لتصبحمركزًا معترفًا به عالميًا للتميز األكاديمي واالبتكار . وعلى مدار أكثر من ستة عقود من الريادة التعليمية، خرّجت الجامعة أكثر من 150,000 خريج يعملون في مختلف الصناعات والمجاالت األكاديمية حول العالم. الجامع

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QUESTION:
هل تقدم الجامعة برامج دراسية عبر الإنترنت؟

FINAL ANSWER
----------------------------------------------------------------------------------------------------
نعم، تقدم الجامعة برامج دراسية عبر الإنترنت. حيث أن هناك إشارة واضحة إلى تنظيم شبكة خريجي الجامعة ندوات عبر الإنترنت كل ثالثة أشهر، وهذا يشير إلى تقديم الجامعة برامج دراسية عبر الإنترنت.

RETRIEVED EVIDENCE
----------------------------------------------------------------------------------------------------

Rank: 1 | Chunk ID: 9 | Similarity Score: 0.8300
----------------------------------------------------------------------------------------------------
العالم. وتنظم شبكة خريجي جامعة THU ندوات عبر اإلنترنت كل ثالثة أشهر، وبرامج لإلرشاد والتوجيه، ولقاءات للخريجين. نهاية المستند

Rank: 2 | Chunk ID: 3 | Similarity Score: 0.8123
----------------------------------------------------------------------------------------------------
في علوم الحاسوب (BSc in Computer Science) • بكالوريوس اآلداب في العالقات الدولية (BA in Interna

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QUESTION:
هل توجد مساعدات مالية للطلاب الدوليين؟

FINAL ANSWER
----------------------------------------------------------------------------------------------------
نعم، يوجد مساعدات مالية للطلاب الدوليين. يتم تقديم منح قائمة على الحاجة المالية للطلاب الدوليين الذين يحتاجون إلى مساعدة مالية. كما يتم تقديم منح التفوق التي تغطي من 25% إلى 100% من الرسوم الدراسية.

RETRIEVED EVIDENCE
----------------------------------------------------------------------------------------------------

Rank: 1 | Chunk ID: 6 | Similarity Score: 0.7916
----------------------------------------------------------------------------------------------------
حسين (Mr. Samer Hussein) مكتب التسجيل (Registrar's Office): يتولى مسؤولية السجالت األكاديمية، وتسجيل المقررات الدراسية، وكشوف الدرجات. مكتب البرامج الدولية (Office of International Programs): يساعد الطالب الدوليين ويتولى الشراكات الدولية. 6. البحث العلمي واالبتكار • مركز أبحاث الطاقة المتجددة • مختبر الذكاء االصطناعي والروبوتات • معهد السياسات العامة والتنمية • 